# Stochastic transfer, replicated and paired

The stochastic collapse is the last single-run number carrying weight. It
was one adapter at one anchor draw: 7/32 stochastic against 32/32
deterministic. The replication run has since shown that single
deterministic numbers are not safe, so this one needs the same treatment.

Each run trains one adapter and evaluates it on **both** the deterministic
and the stochastic instances, so the drop is paired within adapter rather
than compared across runs.

| condition | anchors | targets on the optimal route | deterministic |
| --- | --- | --- | --- |
| **A** | bespoke `anchors.py` | 4/20 | 26, 28, 31 |
| **C** | `anchors_v22`, four `irrelevant` per `silent_break` | 4/20 | 24, 25, 26 |

C is included because it is the generator worth keeping: same result as
the bespoke one on deterministic instances, tighter across draws, and
built from the frozen environment. Whether it also collapses under noise
is a separate question from whether A does, and the answer changes what
goes in the writeup.

## Why this is the right test

The evidence-only rule "the pair that never succeeds in period B" scores
32/32 on the stochastic instances, so the task is exactly as solvable.
What changes is that self-loops stop being diagnostic: a healthy link
self-loops whenever an attempt fails, so roughly twelve pairs per period
self-loop at least once. A model reading "the pair that self-loops in B"
should collapse; one reading "the pair that never succeeds in B" should
not.

* **Both collapse** -> both generators teach the surface form, and 6.2
  says so with replicates behind it.
* **C holds and A collapses** -> off-route training teaches something
  more robust, which is a stronger claim than the draft currently makes.
* **Neither collapses** -> the original 7/32 was a draw artefact and the
  mechanism paragraph comes out.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, gc, importlib.util, collections, re
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG   = "storeplicate"
REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
GEN_PATH  = find_dir("gen_payloads.py")
_old = [p for p in glob.glob("/kaggle/input/**/anchors.py", recursive=True)
        if os.path.basename(p) == "anchors.py"]
if not _old:
    raise SystemExit("the original anchors.py is not attached; condition A "
                     "needs it. Add it to the ecpm eval dataset.")
OLD_ANCHORS = _old[0]
OUT_DIR = "/kaggle/working"
for label, p in (("repo", REPO_PATH), ("eval", EVAL_PATH),
                 ("gen_payloads", GEN_PATH), ("old anchors", OLD_ANCHORS)):
    print(f"{label:14} {p}")

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel and run from the "
    "first cell")
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8          # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"\ngpu: {torch.cuda.get_device_name(0)} | dtype: {DTYPE}")

In [ ]:
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
CONDITIONS  = ["A_bespoke", "C_offroute"]   # drop C to halve the runtime
DRAWS       = [1000, 2000, 3000]
N_WORLDS    = 40
K           = 5
TOTAL_STEPS = 105
LR          = 1e-4
GRAD_ACCUM  = 4
MAX_LEN     = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]
MIX = {"A_bespoke": None,
       "C_offroute": ("irrelevant",) * 4 + ("silent_break",)}

sys.path.insert(0, EVAL_PATH)
sys.path.insert(0, GEN_PATH)
import anchors_v22, gen_payloads as GP, ecpm_eval as E
rp = anchors_v22.load_env(REPO_PATH)
E.attach(REPO_PATH)
_s = importlib.util.spec_from_file_location("old_anchors", OLD_ANCHORS)
oa = importlib.util.module_from_spec(_s)
_s.loader.exec_module(oa)
print(f"{len(CONDITIONS) * len(DRAWS)} runs, each trained once and evaluated "
      "on deterministic and stochastic instances")

## Instances

Both sets are built once and shared by every run, so the adapter is the
only thing that differs. The seed lists are matched, so the paired drop
is over the same graphs.

In [ ]:
rp_gp, _ = GP.load_env(REPO_PATH)

def build_instances(deterministic, n=32):
    out = []
    for seed in range(80):
        if len(out) >= n:
            break
        try:
            out.append(GP.build_for_seed(rp_gp, seed, deterministic, K,
                                         "silent_break"))
        except (ValueError, RuntimeError):
            pass
    return out

DET = build_instances(True)
STO = build_instances(False)
assert [p["seed"] for p in DET] == [p["seed"] for p in STO], \
    "seed lists differ; the drop would not be paired"
print(f"{len(DET)} instances each: {[p['seed'] for p in DET]}")

# how badly is the surface cue broken in the stochastic set?
TRIPLE = re.compile(r"\(([A-Z]), (a\d+), ([A-Z])\)")
loops, unique_dead = [], 0
for p in STO:
    post = p["single"]["detection"].split("Observations, period B:", 1)[1]
    loops.append(len({(u, a) for u, a, v in TRIPLE.findall(post) if u == v}))
    ok, seen = collections.Counter(), collections.Counter()
    for u, a, v in TRIPLE.findall(post):
        seen[(u, a)] += 1
        ok[(u, a)] += (u != v)
    dead = [q for q in seen if ok[q] == 0]
    unique_dead += (len(dead) == 1
                    and dead[0] == tuple(p["facts"]["target_pair"]))
print(f"stochastic period B: median {sorted(loops)[len(loops)//2]} pairs "
      f"self-loop at least once")
print(f"target is the uniquely never-succeeding pair: {unique_dead}/{len(STO)}")
print("-> surface cue broken, robust cue intact, ceiling still 32/32")

## Anchors, training and evaluation

The base is reloaded per run rather than unloaded in place, so no run's
LoRA state can leak into the next.

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_examples(cond, draw):
    if cond == "A_bespoke":
        worlds = oa.build_anchors(n_worlds=N_WORLDS, k=K, first_seed=draw)
        ex = [{"prompt": w["evidence"] + "\n\n" + q, "gold": a}
              for w in worlds for _, q, a in w["items"]]
        on = off = 0
        for w in worlds:
            if not w["changed"]:
                continue
            adj = oa.make_world(w["seed"])
            route = {(s["node"], s["action"])
                     for s in oa.shortest(adj, oa.START, oa.GOAL, None)}
            if tuple(w["broken"]) in route:
                on += 1
            else:
                off += 1
    else:
        anchors_v22.CHANGED_CONDITIONS = MIX[cond]
        worlds = anchors_v22.build_anchor_set(
            rp, n_worlds=N_WORLDS, k=K, stochastic_share=0.0,
            first_seed=draw, preservation_changed_repeat=1)
        ex = anchors_v22.to_examples(worlds)
        on = off = 0
        for w in worlds:
            if not w["changed"]:
                continue
            rec = rp.build_record(
                anchors_v22._scenario(w["seed"], w["condition"], K), True)
            if rec["change"].get("on_optimal_route"):
                on += 1
            else:
                off += 1
    assert len(ex) == 140, f"{cond}/{draw}: {len(ex)} examples"
    assert min(w["seed"] for w in worlds) > 79
    return ex, {"on_route": on, "off_route": off}

def as_ids(x):
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def encode(e):
    pre = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": e["prompt"]}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(e["gold"] + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": pre + ans, "labels": [-100] * len(pre) + ans}

class Rows(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return dict(self.rows[i])

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)

def run_one(cond, draw, seed_idx):
    ex, stats = build_examples(cond, draw)
    enc = [encode(e) for e in ex]
    assert max(len(x["input_ids"]) for x in enc) <= MAX_LEN

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=BNB, device_map={"": 0})
    model = prepare_model_for_kbit_training(model,
                                            use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
    res = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"{OUT_DIR}/sr_tmp",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=GRAD_ACCUM,
            max_steps=TOTAL_STEPS, learning_rate=LR,
            warmup_steps=max(1, TOTAL_STEPS // 20),
            lr_scheduler_type="cosine", logging_strategy="no",
            save_strategy="no", report_to=[], seed=42 + seed_idx,
            bf16=USE_BF16, fp16=not USE_BF16,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}),
        train_dataset=Rows(enc), data_collator=collate).train()

    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True

    def gen(messages, n):
        e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                    return_tensors="pt", return_dict=True)
        e = {k: v.to(model.device) for k, v in e.items()}
        with torch.no_grad():
            o = model.generate(**e, max_new_tokens=n, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        return tok.decode(o[0, e["input_ids"].shape[1]:],
                          skip_special_tokens=True)

    out, allrows = {}, []
    for mode, pays in (("det", DET), ("sto", STO)):
        rows = E.run_arm(pays, gen, arm="arm_c", mode="single",
                         probes=["localization"], verbose=False)
        for r in rows:
            r["condition"], r["draw"], r["eval_mode"] = cond, draw, mode
        allrows += rows
        out[mode] = {
            "localize": sum(bool(r["scored"].get("correct")) for r in rows),
            "node": sum(1 for r in rows
                        if r["parsed"].get("node") == r["target"][0]),
            "n": len(rows)}

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {"condition": cond, "draw": draw, **stats,
            "det": out["det"], "sto": out["sto"],
            "final_loss": res.metrics.get("train_loss")}, allrows

In [ ]:
res_path = os.path.join(OUT_DIR, f"results_{RUN_TAG}.jsonl")
raw_path = os.path.join(OUT_DIR, f"raw_{RUN_TAG}.jsonl")
results, done = [], set()
if os.path.exists(res_path):
    for line in open(res_path):
        r = json.loads(line)
        results.append(r)
        done.add((r["condition"], r["draw"]))
    print(f"resuming, {len(results)} runs already saved")

t_all = time.time()
for si, draw in enumerate(DRAWS):
    for cond in CONDITIONS:
        if (cond, draw) in done:
            print(f"{cond} draw {draw}: done")
            continue
        t0 = time.time()
        summary, rows = run_one(cond, draw, si)
        with open(res_path, "a") as f:
            f.write(json.dumps(summary) + "\n")
        with open(raw_path, "a") as f:
            for r in rows:
                f.write(json.dumps(r) + "\n")
        results.append(summary)
        print(f"{cond:12} draw {draw}: det {summary['det']['localize']}/32  "
              f"sto {summary['sto']['localize']}/32  "
              f"loss {summary['final_loss']:.4f}  "
              f"({(time.time()-t0)/60:.1f} min)")
print(f"\ntotal {(time.time()-t_all)/60:.1f} min")

## Results

In [ ]:
import pandas as pd
import statistics as st

df = pd.DataFrame([{"condition": r["condition"], "draw": r["draw"],
                    "on_route": f"{r['on_route']}/"
                                f"{r['on_route']+r['off_route']}",
                    "det": r["det"]["localize"], "sto": r["sto"]["localize"],
                    "drop": r["det"]["localize"] - r["sto"]["localize"],
                    "sto_node": r["sto"]["node"],
                    "loss": round(r["final_loss"], 4)}
                   for r in results]).sort_values(["condition", "draw"])
display(df)

print(f"\n{'condition':13} {'det':>16} {'sto':>16} {'mean drop':>10}")
for cond in CONDITIONS:
    sub = [r for r in results if r["condition"] == cond]
    if not sub:
        continue
    d = [r["det"]["localize"] for r in sub]
    s = [r["sto"]["localize"] for r in sub]
    print(f"  {cond:11} {str(sorted(d)):>16} {str(sorted(s)):>16} "
          f"{st.mean(d) - st.mean(s):>10.1f}")

print("\nreference on the stochastic instances:")
print("  evidence-only rule, uniquely dead in B   32/32")
print("  untrained, evidence in prompt             3/32")
print("  single earlier run of this adapter        7/32")

json.dump(results, open(os.path.join(OUT_DIR, f"summary_{RUN_TAG}.json"), "w"),
          indent=1)

In [ ]:
# does every adapter drop, or only some?
print("paired drop per run (det -> sto):")
for r in sorted(results, key=lambda r: (r["condition"], r["draw"])):
    d, s = r["det"]["localize"], r["sto"]["localize"]
    bar = "#" * max(0, d - s)
    print(f"  {r['condition']:12} draw {r['draw']}  {d:>2} -> {s:>2}  "
          f"{bar}")
drops = [r["det"]["localize"] - r["sto"]["localize"] for r in results]
print(f"\nall {len(drops)} runs dropped: {all(x > 0 for x in drops)}")
print(f"smallest drop {min(drops)}, largest {max(drops)}")
print("\nA consistent drop across every draw is the surface-rule reading.")
print("If one condition holds up where the other does not, off-route")
print("training teaches something that survives noise, which is a stronger")
print("claim than the draft currently makes.")

## Package

In [ ]:
import zipfile, shutil
shutil.rmtree(f"{OUT_DIR}/sr_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if (not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path
                or ".ipynb_checkpoints" in path):
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")